# The Anatomy of Global Trade — Explainer Notebook
### Project Assignment B · "The Viz and the Notebook"
**Course:** Social Data Analysis and Visualization · DTU  
**Data story website:** [link to GitHub Pages site](https://emmbr.github.io/github.io/) 

**Data source:** World Bank WITS API · UNSD Comtrade · 1988–2023

---
## 1. Motivation

### What is the dataset?

The dataset is drawn from the **World Bank World Integrated Trade Solution (WITS)** database, accessed via the `world_trade_data` Python API. It covers:

- **Export and import product shares (%)** per country per HS sector group (16 sectors), annually from 1988 to 2023
- **163 reporting countries**, with World aggregate available as a single reporter (`wld`)
- **Sector groups** follow the Harmonized System (HS) classification: Animal, Vegetable, Food Products, Minerals, Fuels, Chemicals, Plastics, Hides, Wood, Textiles, Footwear, Stone/Glass, Metals, Machinery & Electronics, Transport, Miscellaneous

Additionally: the **Herfindahl-Hirschman Index (HHI)** for export market concentration per country, and **MFN/AHS tariff indicators** via the WITS `DF_WITS_TradeStats_Tariff` dataflow.

### Why this dataset?

Three reasons:

1. **Breadth and depth.** 35 years of data across 163 countries and 16 sectors allows both time-series narratives (how has the world changed?) and cross-sectional comparisons (who trades what?).

2. **Accessibility.** The WITS API enables reproducible, automated data collection — no manual CSV downloads. This makes the analysis auditable and updatable.

3. **Denmark as a focal lens.** As a small, highly open economy (exports ≈ 71% of GDP, Statistics Denmark), Denmark is an interesting case study: it punches above its weight in food and pharmaceutical exports despite having no fossil fuel reserves comparable to Norway.

### Goal for the end user's experience

The target reader is a **non-specialist with general interest in economics or global affairs** — think a fellow student who hasn't taken this course. The goal is to:

- Provide an **establishing overview** (what does global trade look like?) before zooming into specifics
- Allow **interactive exploration** — the geo heatmap sector slider and radar country comparison let the reader ask their own questions
- Leave the reader with **three clear takeaways**: Machinery dominates, Fuels is volatile, and Denmark's food identity is structurally deep

The website follows the **magazine genre** (Segel & Heer, 2010): long-form scrolling narrative with interleaved visualizations, pull quotes, and captions — author-driven flow with reader-driven exploration within each chart.

---
## 2. Basic Stats

### Data cleaning and preprocessing

The raw WITS API returns a **MultiIndex DataFrame** with levels: `['Freq', 'Reporter', 'Partner', 'ProductCode', 'Indicator', 'Year']`.

Key preprocessing decisions:

1. **Filter to `Partner = 'World'`** — we focus on each country's relationship with the global market, not bilateral trade pairs. This keeps the analysis tractable and comparable across countries.

2. **Filter to `Freq = 'Annual'`** — the API also returns sub-annual data for some indicators; we keep only annual observations.

3. **Year index converted to `int`** — the raw API returns years as strings, which breaks matplotlib x-axis rendering and `fill_between`.

4. **Year-by-year loading for all-reporter data** — calling `reporter='all'` with `year='all'` silently returns only the most recent year (an undocumented API limitation). The cross-country dataset is therefore assembled by looping over eight snapshot years explicitly: `[1995, 2000, 2005, 2010, 2015, 2018, 2020, 2022]`, making 256 API calls (8 years × 16 sectors × 2 indicators).

5. **Sector alignment via `.intersection()`** — export and import DataFrames are aligned on common products and years before computing the trade balance, so missing sectors for one flow don't silently produce NaN balances.

6. **Normalised trade balance instead of raw difference** — the trade balance is computed as `(X − M) / (X + M)` rather than the raw percentage-point difference `X − M`. Raw differences are sector-size dependent (a 5 pp balance in Machinery, which accounts for 26% of trade, means something very different from 5 pp in Footwear at 0.5%). The normalised form maps to ±1 regardless of sector scale, making cross-sector and cross-country comparisons valid.

7. **First-differencing for correlation** — raw export shares are non-stationary (trending), so Pearson correlations are computed on year-on-year changes (`df.diff().dropna()`) to avoid spurious correlations (cf. Granger & Newbold, 1974).

In [ ]:
# Dataset shape and coverage
print(f"World export share matrix: {wld_export_df.shape}")
print(f"  → {wld_export_df.shape[0]} years × {wld_export_df.shape[1]} sectors")
print(f"  → Year range: {wld_export_df.index.min()} – {wld_export_df.index.max()}")
print(f"\nAll-reporter cross-section: {exp_cross.shape[0]} countries")
print(f"Country time series loaded: {list(trend_data.keys())}")
print(f"\nMissing values (world export): {wld_export_df.isnull().sum().sum()} cells")

### Key exploratory findings

| Observation | Value |
|---|---|
| Largest sector (2023) | Machinery & Electronics — 26.3% of world exports |
| Most volatile sector | Fuels — std dev ≈ 3.2 pp across years (3× next highest) |
| Most stable sector | Footwear — std dev < 0.3 pp |
| Strongest positive correlation | Animal ↔ Food Products (r = +0.85) |
| Strongest negative correlation | Chemicals ↔ Fuels (r = −0.68) |
| Denmark's biggest overweight vs world | Animal products (~3× world average share) |
| Countries with data (cross-section) | 163 (ISO3-matched); 206 reporters in raw data |
| Years covered (trade balance) | 8 snapshots: 1995, 2000, 2005, 2010, 2015, 2018, 2020, 2022 |
| Time span (global series) | 36 years (1988–2023) |
| World avg MFN tariff trend | Declining 1990–2018; plateau/uptick post-2018 |
| Most protected sector (MFN) | Animal / Vegetable / Food — approx. 3–4× higher than Machinery |

In [ ]:
# Summary statistics — world export shares
print("=== World Export Share Summary (%) ===")
print(wld_export_df.describe().round(2))

print("\n=== Latest year sector ranking ===")
print(wld_export_df.iloc[-1].sort_values(ascending=False).round(2))

---
## 3. Data Analysis

### What we learned from the data

**Global composition (Chapter 1)**  
The stacked area chart reveals three distinct eras:
- **1988–2000:** Commodity-heavy basket; Fuels and raw materials together exceeded 30%
- **2001–2014:** Machinery rises as China enters global supply chains; first Fuels spike in 2008
- **2015–2023:** Machinery consolidates dominance; second Fuels spike in 2022 (Ukraine war); intermediate goods share begins to decline (WTO, 2023: fell to 48.5% in H1 2023)

**Sector correlations (Chapter 1)**  
First-differenced correlations reveal two structural groupings:
- **Commodity cluster:** Animal, Vegetable, Food Products, Wood move together (positive r) — driven by shared agri-commodity demand cycles
- **Industrial vs energy tension:** Machinery/Chemicals are negatively correlated with Fuels — when energy prices spike (raising input costs), manufacturing output share contracts

**Trade balance geography (Chapter 2)**  
The normalised trade balance `(X − M) / (X + M)` is used rather than the raw percentage-point difference, so that a country with a small absolute trade volume but total specialisation in a sector (e.g. a small oil state in Fuels) reads as +1, not a fraction of what a large country shows. This makes the 206-country cross-section genuinely comparable.

The Fuels choropleth is the most polarised map: Gulf states, Russia, Norway, Angola are deep-blue net exporters; Europe and East Asia are uniformly red net importers. The Machinery map is nearly the exact inverse — textbook Heckscher-Ohlin specialisation (Krugman & Obstfeld, 2018). Animating the year slider from 1995 to 2022 shows China's Machinery position shifting from neutral to strongly blue over the period — the visual counterpart of Autor (2021)'s finding on China's manufacturing upgrade.

**Structural trends (Chapter 2)**  
OLS trend slopes show China's Machinery share rising fastest in the dataset, consistent with Autor (2021) on China's "graduation" from labour-intensive to capital-intensive manufacturing. Denmark's Animal/Food trend is flat — a structural stability that Henriksen (2006) traces to 19th-century co-operative institutions.

**Tariff analysis (Chapter 3)**  
The global average MFN simple-average rate has declined substantially since 1995, with the steepest drop coinciding with China's WTO accession in 2001. However, the decline is uneven across sectors: agricultural categories (Animal, Vegetable, Food) remain 3–4× more protected than Machinery and Electronics, reflecting the failure of the Doha Development Round to achieve agricultural liberalisation. After 2018, the global average flattens and in some sectors nudges upward, consistent with US–China tariff escalation and the broader "slowbalisation" pattern identified in recent WTO reviews. Cross-referencing with the trade balance maps reveals the policy paradox: the sectors where most countries are net importers (food and agriculture for East Asia and Europe) face the highest barriers, while the sectors where manufacturing exporters hold their strongest surpluses (Machinery, Chemicals) face near-zero tariffs.

**Denmark in context (Chapter 4)**  
The radar chart confirms Denmark over-indexes in Animal (+190% vs world avg), Food (+80%), and Chemicals (+40%, driven by Novo Nordisk). It under-indexes in Transport and Metals. Compared to Germany, Denmark is more commodity-oriented; compared to Norway, far less fuel-dependent. The tariff chapter adds a policy lens to this: Denmark's two strongest export sectors (Animal, Food) operate in the highest-tariff global environment, making Danish agri-food competitiveness — achieved through scale, processing quality, and FTA access — especially notable.

In [ ]:
# Reproduce the slope analysis
def compute_slope(series):
    s = series.dropna()
    if len(s) < 5: return float('nan')
    x = (lambda a: (a - a.mean()) / a.std())(s.index.astype(float).values)
    return float(np.polyfit(x, s.values, 1)[0])

slope_df = pd.DataFrame({
    country: {col: compute_slope(df[col]) for col in df.columns}
    for country, df in trend_data.items()
}).T

print("Top 5 rising sector trends across all countries:")
print(slope_df.stack().sort_values(ascending=False).head(10))
print("\nTop 5 declining sector trends:")
print(slope_df.stack().sort_values().head(10))

---
## 4. Genre

### Genre: Magazine Style (Segel & Heer, 2010)

The website uses the **magazine style** genre — a long-form scrolling narrative where text and visualizations alternate, with the author controlling the sequence of revelations. This suits our goal: the story has a clear arc (global → geographic → temporal → national) that benefits from guided reading rather than free exploration.

---

### Figure 7 tools used — Visual Narrative

| Category | Tool used | How |
|---|---|---|
| **Visual structuring** | Consistent colour scheme (dark background, blue=export, red=import) across all charts | Reader builds colour intuition once; applies it across all 6 charts |
| **Highlighting** | Accent bars in the sector bar chart (above-median sectors highlighted in blue) | Directs attention to dominant sectors immediately |
| **Transition guidance** | Scroll-reveal animation (IntersectionObserver) with staggered fade-in | Paces reading; prevents information overload on page load |

---

### Figure 7 tools used — Narrative Structure

| Category | Tool used | How |
|---|---|---|
| **Ordering** | Linear chapter progression (Big Picture → Geography → Trends → Denmark) | Author-imposed sequence following establishing shot → close-up → feature extraction logic |
| **Interactivity** | Sector slider on geo heatmap; country dropdown on radar | Reader-driven exploration within each chapter; Plotly `updatemenus` + `animation_frame` |
| **Messaging** | Pull quotes, callout boxes, insight cards, chapter intros | Each visualization is preceded by framing text and followed by a caption — redundant encoding reinforces key findings |

---
## 5. Visualizations

### Chart choices and justification

**Fig 1.1 — Stacked area chart (global composition over time)**  
*Why:* The stacked area is the canonical form for part-to-whole evolution over time. It immediately shows both the absolute share of each sector and how the whole basket shifts. Alternative (small multiples) would hide the compositional story. Interactive hover on `x unified` lets the reader inspect exact values without cluttering the chart.

**Fig 1.2 — Horizontal bar chart (latest year snapshot)**  
*Why:* A ranked bar chart answers "what's biggest right now?" faster than any other form. Horizontal orientation accommodates long sector labels without rotation. Colour encodes above/below median — a feature-extraction device (Segel & Heer, 2010).

**Fig 1.3 — Correlation heatmap (lower triangle)**  
*Why:* A heatmap is the standard form for symmetric matrices. Lower triangle only removes redundancy. First-differenced inputs make the correlations economically meaningful rather than spurious (Granger & Newbold, 1974). The RdBu diverging scale maps naturally to positive/negative.

**Fig 2.1 — Choropleth with year slider + sector dropdown (normalised trade balance)**  
*Why:* Geographic data belongs on a map. The year animation slider lets one map serve eight time periods; the sector dropdown covers 16 product groups — a single interactive figure replaces 128 static maps. The normalised metric `(X − M) / (X + M)` is chosen over the raw pp difference because it removes sector-size effects: a ±1 scale makes Angola's Fuels surplus directly comparable to Denmark's Animal surplus, which a raw pp comparison cannot do. Plotly's `go.Figure` with one trace per sector and frames per year is used rather than `px.choropleth` because Plotly Express cannot animate over both a frame dimension (year) and a trace-visibility dimension (sector) simultaneously. RdBu with midpoint at 0 maps directly to the export-minus-import sign.

**Fig 3.1 — Trend heatmap (country × sector)**  
*Why:* This is a matrix of time-series summaries. A heatmap compresses 8×16 = 128 time series into a single scannable view. Colour encodes direction and magnitude simultaneously. Alternative (line chart grid) would require 128 panels — unreadable.

**Fig 4.1 — Two-panel MFN tariff figure (static matplotlib)**  
*Why:* The tariff data comes from WITS SDMX rather than the main WITS REST API, so it is assembled separately and saved as a static PNG. A static figure is appropriate here: the story is a trend line with milestone markers (left panel) and a ranked bar (right panel) — neither requires interaction. The two-panel layout allows the time-series story (how rates have changed) and the cross-section story (which sectors remain protected) to be read together. Milestone annotations use staggered heights and boxed labels to prevent overlap on the vertical lines.

**Fig 5.1 — Radar chart with country dropdown**  
*Why:* A radar/spider chart is ideal for comparing a multi-dimensional profile against a baseline. Each axis is a sector; the shape of the polygon encodes the export identity. The world average overlay makes over/under-indexing immediately visible. The pure-Plotly dropdown (no ipywidgets) ensures it works in a static GitHub Pages deployment.

---
## 6. Discussion

### What went well

- **API pipeline is fully reproducible.** The entire dataset is fetched programmatically from WITS — no manual downloads, no CSVs committed to the repo.
- **Visual consistency.** The dark theme, typography (Georgia/Playfair Display + JetBrains Mono), and blue/red colour language are applied consistently across the Python charts and the HTML page, creating a coherent visual platform (Segel & Heer, 2010).
- **The geo heatmap is the standout.** The year slider + sector dropdown compress 128 static maps into one interactive figure — it's the chart readers spend the most time with.
- **Normalised trade balance.** Switching from raw pp to `(X − M) / (X + M)` makes cross-sector comparison valid and the ±1 colorscale self-explanatory. This was a substantive methodological improvement, not just a cosmetic one.
- **Tariff analysis completed.** The WITS SDMX endpoint (`DF_WITS_TradeStats_Tariff`, `MFN-SMPL-AVRG`, `reporter=WLD`) yields global average tariff trends 1990–2022 that add the policy dimension missing from trade-flow analysis alone.
- **References are primary.** WTO Statistical Reviews, NBER working papers, and Statistics Denmark are cited rather than secondary aggregators.

### What is still missing / could be improved

| Gap | Why it matters | Status / Potential fix |
|---|---|---|
| **Year-by-year API limitation** | `reporter='all'` + `year='all'` silently returns only the most recent year — a non-obvious bug that produced a single-year choropleth. Fixed by explicit year loop. | **Fixed** — 8 snapshot years, 256 API calls, 1995–2022 coverage |
| **Normalised trade balance** | Raw pp balance is sector-size dependent; cross-sector comparisons were invalid. | **Fixed** — `(X − M) / (X + M)` implemented |
| **Tariff data** | Overlaying MFN tariff rates adds the policy dimension to the trade-flow story. | **Done** — WITS SDMX `MFN-SMPL-AVRG` for `reporter=WLD`, plotted as two-panel figure |
| **More countries in radar** | Only 8 countries loaded in `trend_data` due to API rate limits | Load 20–30 countries with pickle caching; expand dropdown options |
| **Statistical significance** | Trend slopes are OLS point estimates with no confidence intervals reported | Add bootstrap CIs or at minimum report R² alongside slope |
| **Tariff × balance overlay** | Combining the MFN rate with the trade balance on a single map would make the protection-specialisation paradox directly visible | Feasible: join `tariff_global` sector rates onto `bal_long_all` by sector and year; encode as bubble size or secondary colour |
| **Mobile layout** | The magazine layout is desktop-first; on small screens iframes become very narrow | Set explicit iframe min-heights and use CSS `aspect-ratio` for chart containers |

---
## 7. Contributions

*Note: This project was completed individually. All sections were produced by a single author.*

| Component | Responsible |
|---|---|
| Data pipeline (WITS API, helpers, caching) | Emma |
| Exploratory analysis (stacked area, bar, correlation) | Emma |
| Cross-country analysis (geo heatmap, trend heatmap) | Emma |
| Denmark focus (radar chart, peer comparison) | Emma |
| Website design and narrative (index.html) | Emma |
| References and academic framing | Emma |


---
## 8. References

- **Autor, D. (2021).** China's export prowess and the opening of market opportunities for labor-intensive manufactures. *NBER Working Paper 28313.* https://www.nber.org/papers/w28313

- **Balassa, B. (1965).** Trade liberalisation and revealed comparative advantage. *The Manchester School of Economic and Social Studies, 33*(2), 99–123.

- **Danish Agriculture & Food Council (2023).** *Facts & Figures: Denmark — A Food and Farming Country.* Landbrug & Fødevarer. https://agricultureandfood.dk

- **Granger, C. W. J. & Newbold, P. (1974).** Spurious regressions in econometrics. *Journal of Econometrics, 2*(2), 111–120.

- **Henriksen, I. (2006).** An economic history of Denmark. *EH.net Encyclopedia*, ed. R. Whaples. https://eh.net/encyclopedia/an-economic-history-of-denmark/

- **Krugman, P. R. & Obstfeld, M. (2018).** *International Economics: Theory and Policy* (11th ed.). Pearson. [Heckscher-Ohlin framework, Ch. 5]

- **Mwouts, J.-M. (2019).** *world_trade_data: World Integrated Trade Solution (WITS) API in Python.* https://github.com/mwouts/world_trade_data

- **Segel, E. & Heer, J. (2010).** Narrative visualization: Telling stories with data. *IEEE Transactions on Visualization and Computer Graphics, 16*(6), 1139–1148. http://vis.stanford.edu/files/2010-Narrative-InfoVis.pdf

- **Statistics Denmark (Danmarks Statistik).** *External Trade in Goods — Annual Statistics.* https://www.dst.dk/en

- **World Bank (2024).** *World Integrated Trade Solution (WITS).* https://wits.worldbank.org

- **WTO Secretariat (2023).** *World Trade Statistical Review 2023.* Geneva: WTO. https://www.wto.org/english/res_e/publications_e/wtsr_2023_e.htm

- **WTO Secretariat (2023).** *Global Trade Outlook and Statistics Update — October 2023.* Geneva: WTO. https://www.wto.org/english/res_e/booksp_e/gtos_updt_oct23_e.pdf